In [0]:
storage_account = "dmpro2"
access_key = "blkSylna0g9RsyAQUhzDDyJZ4fS4oWfrveP8HO+FYhp0Y1Cv6hcIGjo2UmS/MWA/TWcTWiAHRTy5+AStQPiZYA=="   # hard-coded

spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", access_key)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


catalog_name = 'dm_pro2'


df_bronze = spark.table(f'{catalog_name}.bronze.products')


df_silver = df_bronze.withColumn("ProductName", trim(col("ProductName")))


df_silver = df_silver.withColumn("ProductID", regexp_replace(col("ProductID"), r'[^A-Za-z0-9]', ''))


df_silver.select("Category").distinct().show()
# Anomalies dictionary
anomalies = {
    "HA": "Home Appliances",
}
df_silver = df_silver.replace(to_replace=anomalies, subset=["Category"])
df_silver.select("Category").distinct().show()


df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.products")



df_bronze = spark.read.table(f"{catalog_name}.bronze.customers")


# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)


# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")


df_bronze.show(10)


#handle null values
null_count = df_bronze.filter(col("CustomerID").isNull()).count()
print(null_count)
df_bronze.filter(col("CustomerID").isNull()).show(3)
df_silver = df_bronze.dropna(subset=["CustomerID"])
row_count = df_silver.count()
print(f"Row count after droping null values: {row_count}")


null_count = df_silver.filter(col("Phone").isNull()).count()
print(f"Number of nulls in phone: {null_count}") 
df_silver.filter(col("Phone").isNull()).show(3)


### Fill null values with 'Not Available'
df_silver = df_silver.fillna("Not Available", subset=["phone"])


df_silver = df_silver.withColumn(
    "LoyaltyPoints",
    when(col("LoyaltyPoints").rlike("^[0-9]+$"), col("LoyaltyPoints")).otherwise(None)
)


# Write raw data to the silver layer
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.customers")


+----------------+
|        Category|
+----------------+
|         Apparel|
|     Electronics|
| Home Appliances|
|              HA|
|Home & Furniture|
+----------------+

+----------------+
|        Category|
+----------------+
|         Apparel|
|     Electronics|
| Home Appliances|
|Home & Furniture|
+----------------+

Row count: 27
Column count: 10
+----------+---------+--------+--------------------+--------+-----------+-----------+-------------+--------------------+--------------------+
|CustomerID|FirstName|LastName|               Email|   Phone|       City|    Country|LoyaltyPoints|        _source_file|         ingested_at|
+----------+---------+--------+--------------------+--------+-----------+-----------+-------------+--------------------+--------------------+
|      C001|     John|     Doe|  john.doe@email.com|555-1234|   New York|        USA|          120|abfss://rawdata@d...|2025-12-01 04:45:...|
|      C002|     Jane|   Smith|jane.smith@email.com|555-5678|Los Angeles|   